# E-Commerce ETL Pipeline - Medallion Architecture
**Tools:** Python, Pandas, Parquet
**Layers:** Bronze (Raw) -> Silver (Clean) -> Gold (Business Reports)

## Problem Statement
Simulated real-world issues: Duplicates, Missing prices (NaN), Dirty dates

## Architecture
- data/bronze: Raw ingestion (CSV)
- data/silver: Cleaned deduplicated data (Parquet - optimized)
- data/gold: Aggregated KPIs for BI

In [1]:
import pandas as pd, numpy as np, os
from datetime import datetime, timedelta
import random

os.makedirs('data/bronze', exist_ok=True)
os.makedirs('data/silver', exist_ok=True)
os.makedirs('data/gold', exist_ok=True)

n=5000
orders = pd.DataFrame({
    'order_id': [f'ORD{i}' for i in range(n)],
    'customer_id': [f'CUST{random.randint(1,500)}' for _ in range(n)],
    'product_id': [f'PROD{random.randint(1,50)}' for _ in range(n)],
    'quantity': np.random.randint(1,5,n),
    'price': np.random.choice([299, 499, 999, np.nan], n),
    'order_date': [datetime.now() - timedelta(days=random.randint(0,30)) for _ in range(n)],
    'city': np.random.choice(['Bangalore','Mumbai','Delhi'], n)
})
orders = pd.concat([orders, orders.sample(100)])
orders.to_csv('data/bronze/orders_raw.csv', index=False)
print(f"Done - {len(orders)} rows created")
orders.head()

Done - 5100 rows created


,order_id,customer_id,product_id,quantity,price,order_date,city
0,ORD0,CUST364,PROD23,2,299.0,2026-08-22 16:30:48.298318,Bangalore
1,ORD1,CUST110,PROD38,2,NaN,2026-09-10 16:30:48.298318,Delhi
2,ORD2,CUST226,PROD1,4,499.0,2026-08-26 16:30:48.298318,Bangalore
3,ORD3,CUST78,PROD47,2,999.0,2026-09-07 16:30:48.298318,Mumbai
4,ORD4,CUST101,PROD31,4,999.0,2026-09-09 16:30:48.298318,Delhi


### Transform & Load ###
- Remove duplicates
- Impute missing price with median
- Calculate total_amount
- Convert to Parquet

In [2]:
# TRANSFORM + LOAD - Silver and Gold

df = pd.read_csv('data/bronze/orders_raw.csv')
print(f"Before cleaning: {len(df)} rows")

# TRANSFORM
df = df.drop_duplicates(subset=['order_id'])
df['price'] = df['price'].fillna(df['price'].median())
df['total_amount'] = df['quantity'] * df['price']
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df = df.dropna(subset=['order_date'])

print(f"After cleaning: {len(df)} rows")

# LOAD - Silver (Parquet - very important for interview)
df.to_parquet('data/silver/orders_clean.parquet', index=False)

# LOAD - Gold Reports
daily = df.groupby(df['order_date'].dt.date)['total_amount'].sum().reset_index()
daily.columns = ['date','revenue']
daily.to_csv('data/gold/daily_revenue.csv', index=False)

top = df.groupby('product_id')['quantity'].sum().sort_values(ascending=False).head(5).reset_index()
top.to_csv('data/gold/top_products.csv', index=False)

city = df.groupby('city')['total_amount'].sum().reset_index()
city.to_csv('data/gold/city_performance.csv', index=False)

print("ETL DONE! Check data/silver and data/gold")
daily.head()

Before cleaning: 5100 rows
After cleaning: 5000 rows
ETL DONE! Check data/silver and data/gold


,date,revenue
0,2026-08-16,239197.0
1,2026-08-17,211022.0
2,2026-08-18,249394.0
3,2026-08-19,225930.0
4,2026-08-20,221034.0
